<a href="https://colab.research.google.com/github/lewliu2003-create/flight_pred_ml/blob/main/Keyan_NSDC_Flight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [5]:
flight_data = '/content/drive/MyDrive/solo_flight_work/flights_data.csv'
df_flight_data = pd.read_csv(flight_data)
df_flight_data.head()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,...,dest_wind_gusts_10m,dest_wind_direction_10m,dest_cloud_cover,dest_weather_code,dest_pressure_msl,MAKER,MODEL,SEATS,YEAR_MFR,PLANE_AGE
0,2026,1,1,1,4,2026-01-01,AA,19805,AA,N101NN,...,25.9,274.0,0.0,0.0,1008.4,AIRBUS INDUSTRIE,A321-231,379.0,2013.0,13.0
1,2026,1,1,1,4,2026-01-01,AA,19805,AA,N101NN,...,8.9,236.0,79.0,51.0,1017.5,AIRBUS INDUSTRIE,A321-231,379.0,2013.0,13.0
2,2026,1,1,1,4,2026-01-01,AA,19805,AA,N103NN,...,15.9,246.0,50.0,2.0,1007.3,AIRBUS,A321-231,379.0,2013.0,13.0
3,2026,1,1,1,4,2026-01-01,AA,19805,AA,N103NN,...,3.7,35.0,18.0,0.0,1018.8,AIRBUS,A321-231,379.0,2013.0,13.0
4,2026,1,1,1,4,2026-01-01,AA,19805,AA,N103NN,...,31.5,300.0,4.0,0.0,1010.1,AIRBUS,A321-231,379.0,2013.0,13.0


In [6]:
df_flight_data.shape

(3488373, 73)

In [7]:

df = df_flight_data.copy()
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
TARGET = "ARR_DEL15"

# 1. keep flights that actually flew and have a target
df = df[(df["CANCELLED"] == 0) & (df["DIVERTED"] == 0)]
df = df.dropna(subset=[TARGET])
df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])

# 2. columns that must NOT be features
LEAK = {"DEP_DELAY", "DEP_DELAY_NEW", "ARR_DELAY", "ARR_DELAY_NEW", "DEP_DEL15", "ARR_DEL15",
        "DEP_DELAY_GROUP", "ARR_DELAY_GROUP", "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY",
        "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY", "DEP_TIME", "ARR_TIME", "TAXI_OUT", "TAXI_IN",
        "WHEELS_OFF", "WHEELS_ON", "ACTUAL_ELAPSED_TIME", "AIR_TIME", "CANCELLED", "DIVERTED",
        "CANCELLATION_CODE"}                                   # known only after the flight
IDS = {"YEAR", "QUARTER", "MONTH", "FL_DATE", "OP_CARRIER_AIRLINE_ID", "OP_CARRIER", "TAIL_NUM",
       "OP_CARRIER_FL_NUM", "ORIGIN_AIRPORT_ID", "ORIGIN_AIRPORT_SEQ_ID", "ORIGIN_CITY_MARKET_ID",
       "DEST_AIRPORT_ID", "DEST_AIRPORT_SEQ_ID", "DEST_CITY_MARKET_ID", "FLIGHTS",
       "dep_datetime", "arr_datetime", "CRS_DEP_TIME", "CRS_ARR_TIME"}   # IDs, duplicates, or don't generalize across the time split

# 3. categorical features (only those present in the file)
CAT_CANDIDATES = ["OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "ORIGIN_STATE_ABR", "DEST_STATE_ABR",
                  "DEP_TIME_BLK", "ARR_TIME_BLK", "MAKER", "MODEL"]
CAT = [c for c in CAT_CANDIDATES if c in df.columns]

# 4. numeric features: every remaining numeric column
NUM = [c for c in df.select_dtypes(include="number").columns
       if c not in LEAK | IDS | set(CAT) and c != TARGET]

for c in CAT:
    df[c] = df[c].astype("category")
feats = CAT + NUM
assert not set(feats) & LEAK, "leakage column in features!"

print("CATEGORICAL:", CAT)
print("NUMERIC:", NUM)
print("\nrows:", len(df), "| delay rate:", round(df[TARGET].mean(), 3))

wx_cols = [c for c in NUM if c.startswith(("origin_", "dest_"))]
miss = df[wx_cols].isna().mean().max()
print("missing weather share:", round(float(miss), 3))

CATEGORICAL: ['OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'ORIGIN_STATE_ABR', 'DEST_STATE_ABR', 'DEP_TIME_BLK', 'ARR_TIME_BLK', 'MAKER', 'MODEL']
NUMERIC: ['DAY_OF_MONTH', 'DAY_OF_WEEK', 'CRS_ELAPSED_TIME', 'DISTANCE', 'DISTANCE_GROUP', 'origin_precipitation', 'origin_snowfall', 'origin_snow_depth', 'origin_temperature_2m', 'origin_wind_speed_10m', 'origin_wind_gusts_10m', 'origin_wind_direction_10m', 'origin_cloud_cover', 'origin_weather_code', 'origin_pressure_msl', 'dest_precipitation', 'dest_snowfall', 'dest_snow_depth', 'dest_temperature_2m', 'dest_wind_speed_10m', 'dest_wind_gusts_10m', 'dest_wind_direction_10m', 'dest_cloud_cover', 'dest_weather_code', 'dest_pressure_msl', 'SEATS', 'YEAR_MFR', 'PLANE_AGE']

rows: 3402621 | delay rate: 0.221
missing weather share: 0.005


In [8]:
train = df[df["FL_DATE"] <  "2026-05-01"]
val   = df[(df["FL_DATE"] >= "2026-05-01") & (df["FL_DATE"] < "2026-06-01")]
test  = df[df["FL_DATE"] >= "2026-06-01"]
print(f"train {len(train):,} (Jan–Apr) | val {len(val):,} (May) | test {len(test):,} (June)")

train 2,203,080 (Jan–Apr) | val 604,346 (May) | test 595,195 (June)


In [9]:
!pip install -q xgboost
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

pos_weight = (train[TARGET] == 0).sum() / (train[TARGET] == 1).sum()
print("scale_pos_weight:", round(pos_weight, 2))

model = xgb.XGBClassifier(
    n_estimators=2000, learning_rate=0.05, max_depth=8,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    tree_method="hist", enable_categorical=True, max_cat_to_onehot=1,
    scale_pos_weight=pos_weight,
    eval_metric="auc", early_stopping_rounds=50,
    device="cuda",          # change to "cpu" if no GPU runtime
)
model.fit(train[feats], train[TARGET], eval_set=[(val[feats], val[TARGET])], verbose=100)

p_xgb = model.predict_proba(test[feats])[:, 1]
print("\nTEST (June)  AUC:", round(roc_auc_score(test[TARGET], p_xgb), 3),
      "| PR-AUC:", round(average_precision_score(test[TARGET], p_xgb), 3),
      "| baseline PR-AUC (delay rate):", round(test[TARGET].mean(), 3))
print(classification_report(test[TARGET], (p_xgb > 0.5).astype(int), digits=3))

scale_pos_weight: 3.69
[0]	validation_0-auc:0.65552
[100]	validation_0-auc:0.69415
[200]	validation_0-auc:0.69651
[268]	validation_0-auc:0.69619


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [01:21:39] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



TEST (June)  AUC: 0.703 | PR-AUC: 0.436 | baseline PR-AUC (delay rate): 0.257
              precision    recall  f1-score   support

         0.0      0.826     0.738     0.780    442158
         1.0      0.421     0.552     0.478    153037

    accuracy                          0.690    595195
   macro avg      0.624     0.645     0.629    595195
weighted avg      0.722     0.690     0.702    595195



In [10]:
imp = pd.Series(model.get_booster().get_score(importance_type="gain")).sort_values(ascending=False)
print(imp.head(20).round(1))

origin_weather_code      1846.9
origin_precipitation      522.4
dest_weather_code         504.6
OP_UNIQUE_CARRIER         332.3
dest_precipitation        295.2
DAY_OF_WEEK               242.5
DAY_OF_MONTH              237.9
origin_snowfall           215.6
DEP_TIME_BLK              201.3
origin_snow_depth         187.7
origin_wind_gusts_10m     180.5
origin_temperature_2m     171.1
dest_wind_gusts_10m       152.1
dest_snow_depth           149.0
ARR_TIME_BLK              143.2
DEST                      139.3
ORIGIN                    137.3
ORIGIN_STATE_ABR          117.1
dest_temperature_2m       108.6
origin_pressure_msl       101.6
dtype: float64


In [11]:
import torch, torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# numeric features: drop near-duplicates, turn wind direction into sin/cos
WIND_DIR = [c for c in NUM if c.endswith("wind_direction_10m")]
DROP_NN  = {"DISTANCE_GROUP", "YEAR_MFR"} | set(WIND_DIR)
NN_NUM   = [c for c in NUM if c not in DROP_NN]

def num_block(d):
    X = d[NN_NUM].astype("float32").copy()
    for c in WIND_DIR:
        rad = np.deg2rad(d[c].astype("float32"))
        X[c + "_sin"], X[c + "_cos"] = np.sin(rad), np.cos(rad)
    X["wx_missing"] = d[wx_cols].isna().any(axis=1).astype("float32")
    return X

# fill and scale using TRAINING statistics only
Xtr = num_block(train)
med = Xtr.median()
Xtr = Xtr.fillna(med)
mu, sd = Xtr.mean(), Xtr.std().replace(0, 1)

def prep_num(d):
    return ((num_block(d).fillna(med) - mu) / sd).values.astype("float32")

# categorical features -> integer codes (0 = missing)
cards = [len(df[c].cat.categories) + 1 for c in CAT]
def prep_cat(d):
    return np.stack([d[c].cat.codes.values + 1 for c in CAT], axis=1).astype("int64")

def to_tensors(d):
    return (torch.tensor(prep_num(d), device=device),
            torch.tensor(prep_cat(d), device=device),
            torch.tensor(d[TARGET].values.astype("float32"), device=device))

tr, va, te = to_tensors(train), to_tensors(val), to_tensors(test)
print("numeric inputs:", tr[0].shape[1], "| categorical columns:", len(CAT), "| category sizes:", cards)

device: cuda
numeric inputs: 29 | categorical columns: 9 | category sizes: [14, 357, 357, 53, 53, 20, 20, 40, 131]


In [12]:
class FlightNet(nn.Module):
    def __init__(self, cards, n_num):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(n, min(16, (n + 1) // 2)) for n in cards])
        emb_dim = sum(e.embedding_dim for e in self.embs)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim + n_num, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128),             nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 1))
    def forward(self, xn, xc):
        e = [emb(xc[:, i]) for i, emb in enumerate(self.embs)]
        return self.mlp(torch.cat(e + [xn], dim=1)).squeeze(1)   # raw score (logit)

@torch.no_grad()
def predict(model, data, bs=65536):
    model.eval(); xn, xc, _ = data
    out = [torch.sigmoid(model(xn[i:i+bs], xc[i:i+bs])) for i in range(0, len(xn), bs)]
    return torch.cat(out).cpu().numpy()

In [13]:
torch.manual_seed(42)
net = FlightNet(cards, tr[0].shape[1]).to(device)
pw = ((tr[2] == 0).sum() / (tr[2] == 1).sum()).detach()
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)          # same idea as scale_pos_weight
opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)

BS, PATIENCE = 4096, 3
best, bad, n = 0, 0, len(tr[0])
for epoch in range(30):
    net.train(); perm = torch.randperm(n, device=device); total = 0
    for i in range(0, n, BS):
        idx = perm[i:i+BS]
        if len(idx) < 2: continue                       # BatchNorm needs > 1 row
        opt.zero_grad()
        loss = loss_fn(net(tr[0][idx], tr[1][idx]), tr[2][idx])
        loss.backward(); opt.step()
        total += loss.item() * len(idx)
    auc = roc_auc_score(va[2].cpu().numpy(), predict(net, va))
    print(f"epoch {epoch+1:2d} | train loss {total/n:.4f} | val AUC {auc:.4f}")
    if auc > best:
        best, bad = auc, 0
        torch.save(net.state_dict(), "best_net.pt")
    else:
        bad += 1
        if bad >= PATIENCE:
            print("early stop"); break

net.load_state_dict(torch.load("best_net.pt"))
print("best val AUC:", round(best, 4))

epoch  1 | train loss 0.9994 | val AUC 0.6855
epoch  2 | train loss 0.9786 | val AUC 0.6898
epoch  3 | train loss 0.9704 | val AUC 0.6913
epoch  4 | train loss 0.9641 | val AUC 0.6897
epoch  5 | train loss 0.9585 | val AUC 0.6917
epoch  6 | train loss 0.9543 | val AUC 0.6870
epoch  7 | train loss 0.9503 | val AUC 0.6920
epoch  8 | train loss 0.9469 | val AUC 0.6879
epoch  9 | train loss 0.9435 | val AUC 0.6889
epoch 10 | train loss 0.9410 | val AUC 0.6868
early stop
best val AUC: 0.692


In [14]:
y = test[TARGET].values
p_nn = predict(net, te)

rows = []
for name, p in [("XGBoost", p_xgb), ("Neural net", p_nn), ("Average of both", (p_xgb + p_nn) / 2)]:
    rows.append({"model": name,
                 "AUC": roc_auc_score(y, p),
                 "PR-AUC": average_precision_score(y, p),
                 "recall@0.5": ((p > 0.5) & (y == 1)).sum() / (y == 1).sum()})
print(pd.DataFrame(rows).round(3))
print("baseline PR-AUC:", round(y.mean(), 3))
print("\nNeural net report:\n", classification_report(y, (p_nn > 0.5).astype(int), digits=3))

             model    AUC  PR-AUC  recall@0.5
0          XGBoost  0.703   0.436       0.552
1       Neural net  0.703   0.444       0.554
2  Average of both  0.710   0.450       0.558
baseline PR-AUC: 0.257

Neural net report:
               precision    recall  f1-score   support

         0.0      0.827     0.735     0.778    442158
         1.0      0.420     0.554     0.478    153037

    accuracy                          0.689    595195
   macro avg      0.623     0.645     0.628    595195
weighted avg      0.722     0.689     0.701    595195



In [15]:
idx = torch.randperm(len(tr[0]), device=device)[:300_000]          # sample for speed
tr_sub = (tr[0][idx], tr[1][idx], tr[2][idx])
print("train AUC:", round(roc_auc_score(tr_sub[2].cpu().numpy(), predict(net, tr_sub)), 4))
print("val   AUC:", round(best, 4))

train AUC: 0.7466
val   AUC: 0.692
